# 🎓 Student Placement LPA Predictor
This notebook implements a custom 2-layer Neural Network from scratch using NumPy to predict placement packages (LPA), and hosts an interactive Gradio application.

In [ ]:
import numpy as np
import pandas as pd
!pip install -q gradio

In [ ]:
# Dataset
df = pd.DataFrame([[8,8,4],[7,9,5],[6,10,6],[5,12,7]], columns=['cgpa', 'profile_score', 'lpa'])
df

In [ ]:
def initialize_parameters(layer_dims):
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.ones((layer_dims[l-1], layer_dims[l]))*0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

In [ ]:
def linear_forward(A_prev, W, b):
    return np.dot(W.T, A_prev) + b

def L_layer_forward(X, parameters):
    A = X
    L = len(parameters) // 2
    for l in range(1, L+1):
        A_prev = A
        Wl = parameters['W' + str(l)]
        bl = parameters['b' + str(l)]
        A = linear_forward(A_prev, Wl, bl)
    return A, A_prev

In [ ]:
def update_parameters(parameters,y,y_hat,A1,X):
    parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001 * 2 * (y - y_hat)*A1[0][0])
    parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat)*A1[1][0])
    parameters['b2'][0][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat))

    parameters['W1'][0][0] = parameters['W1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0]*X[0][0])
    parameters['W1'][0][1] = parameters['W1'][0][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0]*X[1][0])
    parameters['b1'][0][0] = parameters['b1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0])

    parameters['W1'][1][0] = parameters['W1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0]*X[0][0])
    parameters['W1'][1][1] = parameters['W1'][1][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0]*X[1][0])
    parameters['b1'][1][0] = parameters['b1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0])

In [ ]:
# Train Network
parameters = initialize_parameters([2,2,1])
epochs = 500 # Increased epochs for perfect alignment
for i in range(epochs):
    Loss = []
    for j in range(df.shape[0]):
        X = df[['cgpa', 'profile_score']].values[j].reshape(2,1)
        y = df[['lpa']].values[j][0]
        y_hat, A1 = L_layer_forward(X, parameters)
        y_hat = y_hat[0][0]
        update_parameters(parameters, y, y_hat, A1, X)
        Loss.append((y-y_hat)**2)
    if (i+1) % 100 == 0 or i == 0:
        print(f'Epoch - {i+1} Loss - {np.array(Loss).mean()}')

In [ ]:
import gradio as gr

def predict_lpa(cgpa, profile_score):
    X_test = np.array([[cgpa], [profile_score]])
    y_pred, _ = L_layer_forward(X_test, parameters)
    predicted_val = float(y_pred[0][0])
    return max(0.0, round(predicted_val, 2))

demo = gr.Interface(
    fn=predict_lpa,
    inputs=[
        gr.Number(label='CGPA', value=8.0, minimum=0.0, maximum=10.0),
        gr.Number(label='Profile Score', value=8.0, minimum=0.0, maximum=20.0)
    ],
    outputs=gr.Number(label='Predicted LPA (Lakhs Per Annum)'),
    title='🎓 Student Placement LPA Predictor',
    description='Enter CGPA and Profile Score to predict package using the custom network.'
)
demo.launch(share=True)